<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-04/week4_tool_use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 (starter): Multi-Tool Assistant

Everything below runs with no API key. The three tools, the validator, the dispatcher, and the dispatch loop are a **worked example** in a toy domain (arithmetic and unit conversion), driven by a scripted list of tool calls. They are the reference, not your submission.

Build your assistant in a domain you choose. The **TODO (you)** comments cover Parts 1 to 4; Part 5 is the submission checklist. The example uses only the Python standard library. For live calls through the OpenAI-compatible endpoint, install the client with `pip install openai`. Setting a key alone does not enable live calls.

In [1]:
import os, ast, operator, json, math
HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)
# The scripted loop below runs either way. Wiring the live model call is yours (Part 1).

GEMINI_API_KEY set: False


### Part 1: Schema Design

For my three data engineering tools, I kept the schemas simple and similar to the starter example. Each tool has a clear purpose, required fields, explicit types, and enums where the possible values are limited. I also used `additionalProperties: False` to prevent the model from passing unexpected arguments.


In [13]:
# Local function definitions with JSON schemas. Adapt these to your model API's tool format.
TOOLS = [
  {'name':'query_data',
   'description':'Query a dataset using a selected operation.',
   'parameters':{'type':'object',
                 'properties':{
                     'dataset':{'type':'string'},
                     'operation':{'type':'string',
                                  'enum':['preview','count','filter']}},
                 'required':['dataset','operation'],
                 'additionalProperties':False}},

  {'name':'validate_data',
   'description':'Check provided data for a selected data quality issue.',
   'parameters':{'type':'object',
                 'properties':{
                     'data':{'type':'array',
                             'items':{'type':'object'}},
                     'check':{'type':'string',
                              'enum':['missing_values','duplicate_ids','schema']}},
                 'required':['data','check'],
                 'additionalProperties':False}},

  {'name':'transform_data',
   'description':'Run an approved data transformation on a dataset. Use expressions such as select(customers), filter(customers), or groupby(customers).',
   'parameters':{'type':'object',
                 'properties':{
                     'expression':{'type':'string'}},
                 'required':['expression'],
                 'additionalProperties':False}},
]

print('tools:', [t['name'] for t in TOOLS])
# TODO (you): replace these three with tools for a domain you pick. Keep the constraint
# discipline: enums where the value set is closed, required fields, explicit types.

tools: ['query_data', 'validate_data', 'transform_data']


## Part 2: Set Up a Guarded Tool

Data transformations can vary widely and become expensive as datasets grow. Restricting the tool to a small set of approved transformations simplifies the operation for the model and keeps execution predictable. In a real data engineering environment, an organization would typically have a set of standard transformations or operations approved for its pipelines rather than allowing arbitrary code to run.

The tool permits only the approved data transformations of selecting, filtering, and grouping data. It blocks arbitrary Python execution, filesystem access, network access, and process execution. A 2-second time limit is also used to reject transformations that take too long to complete. This is a focused safety control, not a full production sandbox.


In [3]:
# Allowlisted data transformation operations with a pre-execution check and time limit.
_OPS = {'select','filter','groupby'}

_DATASETS = {
    'customers': [
        {'id':1,'name':'Alice','state':'MO'},
        {'id':2,'name':'Bob','state':'IL'},
        {'id':3,'name':'Carol','state':'MO'}
    ],
    'orders': [
        {'id':101,'customer_id':1,'amount':50},
        {'id':102,'customer_id':2,'amount':75},
        {'id':103,'customer_id':1,'amount':25}
    ]
}

def _safe(node):
    # Check the entire expression before execution.
    if not isinstance(node, ast.Call):
        raise ValueError('only approved transformation calls are allowed')
    if not isinstance(node.func, ast.Name):
        raise ValueError('only approved transformation calls are allowed')
    if node.func.id not in _OPS:
        raise ValueError('only allowlisted transformations are allowed')
    if len(node.args) != 1 or not isinstance(node.args[0], ast.Name):
        raise ValueError('transformation requires one dataset')
    if node.args[0].id not in _DATASETS:
        raise ValueError('unknown dataset')
    return node.func.id, node.args[0].id

def query_data(dataset, operation):
    if dataset not in _DATASETS: raise ValueError(f'unknown dataset: {dataset}')
    if operation == 'preview': return _DATASETS[dataset][:3]
    if operation == 'count': return len(_DATASETS[dataset])
    if operation == 'filter': return _DATASETS[dataset]
    raise ValueError(f'unsupported query operation: {operation}')

def validate_data(data, check):
    if not isinstance(data, list): raise ValueError('data must be a list')
    rows = data
    if check == 'missing_values':
        return sum(1 for row in rows for value in row.values() if value is None)
    if check == 'duplicate_ids':
        ids = [row['id'] for row in rows if 'id' in row]
        return len(ids) != len(set(ids))
    if check == 'schema':
        return sorted(rows[0].keys()) if rows else []
    raise ValueError(f'unsupported validation check: {check}')

def transform_data(expression):
    start_time = os.times().elapsed

    # Parse and validate the complete expression before execution.
    tree = ast.parse(expression, mode='eval')
    operation, dataset = _safe(tree.body)

    rows = _DATASETS[dataset]

    if operation == 'select':
        output = rows
    elif operation == 'filter':
        output = [row for row in rows if all(value is not None for value in row.values())]
    elif operation == 'groupby':
        groups = {}
        for row in rows:
            key = next(iter(row.values()))
            groups[key] = groups.get(key, 0) + 1
        output = groups

    if os.times().elapsed - start_time > 2:
        raise TimeoutError('transformation exceeded the time limit')

    return output

IMPL = {'query_data':query_data,'validate_data':validate_data,'transform_data':transform_data}

class ToolArgError(Exception): pass

# Validates the tool schemas above.
def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name']==name), None)
    if spec is None: raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict): raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required',[]):
        if r not in args: raise ToolArgError(f'missing required field: {r}')
    for k,v in args.items():
        p = spec['properties'].get(k)
        if p is None: raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str): raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float): raise ToolArgError(f'{k} must be a number (not a boolean)')
            if isinstance(v, float) and not math.isfinite(v): raise ToolArgError(f'{k} must be finite')
        elif p['type'] == 'array':
            if not isinstance(v, list): raise ToolArgError(f'{k} must be an array')
        else:
            raise ValueError(f'extend validate() to support schema type: {p["type"]}')
        if 'enum' in p and v not in p['enum']:
            raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')

def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)
        return {'ok':True,'tool':name,'output':output}
    except ToolArgError as e:
        return {'ok':False,'tool':name,'error_type':'invalid_arguments','message':str(e)}
    except Exception as e:
        return {'ok':False,'tool':name,'error_type':'execution_error','message':str(e)}

print('happy path:', dispatch('query_data', {'dataset':'customers','operation':'preview'}))
print('guarded:', dispatch('transform_data', {'expression':'select(customers)'}))
print('blocked:', dispatch('transform_data', {'expression':'__import__("os").system("echo hi")'}))

happy path: {'ok': True, 'tool': 'query_data', 'output': [{'id': 1, 'name': 'Alice', 'state': 'MO'}, {'id': 2, 'name': 'Bob', 'state': 'IL'}, {'id': 3, 'name': 'Carol', 'state': 'MO'}]}
guarded: {'ok': True, 'tool': 'transform_data', 'output': [{'id': 1, 'name': 'Alice', 'state': 'MO'}, {'id': 2, 'name': 'Bob', 'state': 'IL'}, {'id': 3, 'name': 'Carol', 'state': 'MO'}]}
blocked: {'ok': False, 'tool': 'transform_data', 'error_type': 'execution_error', 'message': 'only approved transformation calls are allowed'}


## Parts 1, 3, and 4: dispatch, evaluation, and recovery
The list below sends an invalid unit, then retries the same conversion with a valid unit. Both calls are scripted. No model chooses these calls or reads the errors, even when a key is set. Use this dispatch example to build your model loop, then evaluate your own tools and document a failure with recovery.

In [4]:
# TODO (you), Part 1: send the tools and query to the model, execute its tool calls, return
# the results with matching call IDs, and let the model continue until it gives an answer.
# TODO (you), Part 3: run at least three queries covering every tool. Include a query that
# needs two tools in sequence, where the second uses the first result. Show the call logs.
# TODO (you), Part 4: find a real failure in your own model's calls. Show the schema, bad
# call, cause, and successful recovery. Explain whether a schema change, description, or retry fixed it.

### Part 1: Live model tool-calling loop
Send the tools and query to the model, execute its tool calls, return the results with matching call IDs, and let the model continue until it gives an answer.

In [5]:
from google.colab import userdata # import API key from secrets

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [6]:
# Check installation and import
# !pip install openai
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get('GEMINI_API_KEY'),
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
)

In [8]:
# Define function to run tools
def run_assistant(query):
    messages = [
        {'role':'user','content':query}
    ]

    while True:
        response = client.chat.completions.create(
            model='gemini-2.5-flash',
            messages=messages,
            tools=[
                {
                    'type':'function',
                    'function':{
                        'name':tool['name'],
                        'description':tool['description'],
                        'parameters':tool['parameters']
                    }
                }
                for tool in TOOLS
            ]
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            print(message.content)
            return

        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)

            print(f'[TOOL] {name}({args})')

            result = dispatch(name, args)

            print(f'[{ "OK" if result["ok"] else "ERR" }] {result}')

            messages.append({
                'role':'tool',
                'tool_call_id':call.id,
                'content':json.dumps(result, allow_nan=False)
            })

### Part 3: Evaluation


In [10]:
## Part 3: Evaluate (live model)

# Query 1: exercise query_data
run_assistant("How many customers are in the customers dataset?")

# Query 2: exercise validate_data
run_assistant("Check the customers dataset for missing values.")

[TOOL] query_data({'operation': 'count', 'dataset': 'customers'})
[OK] {'ok': True, 'tool': 'query_data', 'output': 3}
There are 3 customers in the customers dataset.
The `validate_data` function expects the data to be provided directly as a list of dictionaries, rather than a dataset name. Please provide the customer data you would like to check for missing values.


In [11]:
# Query 3: exercise query_data and validate_data in sequence
run_assistant("Preview the customers dataset and then check the returned data for missing values.")

[TOOL] query_data({'dataset': 'customers', 'operation': 'preview'})
[OK] {'ok': True, 'tool': 'query_data', 'output': [{'id': 1, 'name': 'Alice', 'state': 'MO'}, {'id': 2, 'name': 'Bob', 'state': 'IL'}, {'id': 3, 'name': 'Carol', 'state': 'MO'}]}
[TOOL] validate_data({'data': [{'state': 'MO', 'id': 1, 'name': 'Alice'}, {'name': 'Bob', 'state': 'IL', 'id': 2}, {'state': 'MO', 'name': 'Carol', 'id': 3}], 'check': 'missing_values'})
[OK] {'ok': True, 'tool': 'validate_data', 'output': 0}
I have previewed the customers dataset and checked it for missing values. There are no missing values in the dataset.


In [14]:
# Query 4: exercise transform_data
run_assistant("Select the customers dataset using the data transformation tool.")

[TOOL] transform_data({'expression': 'select(customers)'})
[OK] {'ok': True, 'tool': 'transform_data', 'output': [{'id': 1, 'name': 'Alice', 'state': 'MO'}, {'id': 2, 'name': 'Bob', 'state': 'IL'}, {'id': 3, 'name': 'Carol', 'state': 'MO'}]}
I have selected the customers dataset. Is there anything else I can help you with?


#### Live Run Evaluations

In the first live run, the model correctly selected `query_data` and used the `count` operation on the customers dataset. The tool returned 3 customers, and the model used that result to provide the final answer.

In the second run, I asked the model to check the customers dataset for missing values. Instead of calling `validate_data`, it first called `query_data` with the `count` operation and then explained that `validate_data` required the data to be provided directly. This showed that the model understood the tool's input requirements but did not select the correct tool for the request.

In the final run, the `transform_data` tool initially did not produce a tool call because its description was too general. I updated the description to include examples of valid transformation expressions, such as `select(customers)`, and the model then successfully called the tool. I reran the cell not thinking, without saving the original output, but I had documented the initial result and used it to make the description more specific.


## Part 4: Find One Failure and Explain It

I made four attempts to produce a function-calling failure. The first attempt asked the model to summarize the customers dataset, but it did not make a tool call. The second attempt asked it to check for duplicate IDs, but it again requested that the customer data be provided. The third attempt provided customer data without IDs, and the model successfully called `validate_data` and returned no duplicate IDs. The fourth attempt included a null value in the data, which caused `validate_data` to return a runtime error. This final attempt provided the failure needed for Part 4.


In [15]:
# Part 4 failure test
run_assistant("Summarize the customers dataset.")

I can show you a preview of the customers dataset, but I cannot provide a statistical summary. Would you like to see the preview?


In [16]:
run_assistant("Check the customers dataset for duplicate IDs.")

I can check for duplicate IDs if you provide the customer data.


In [17]:
run_assistant("Check this customer data for duplicate IDs: [{\"name\":\"Alice\"}, {\"name\":\"Bob\"}]")

[TOOL] validate_data({'data': [{'name': 'Alice'}, {'name': 'Bob'}], 'check': 'duplicate_ids'})
[OK] {'ok': True, 'tool': 'validate_data', 'output': False}
There are no duplicate IDs in the customer data.


In [18]:
# The ERROR
run_assistant("Check this data for duplicate IDs: [{\"id\": 1}, null]")

[TOOL] validate_data({'check': 'duplicate_ids', 'data': [{'id': 1}, None]})
[ERR] {'ok': False, 'tool': 'validate_data', 'error_type': 'execution_error', 'message': "argument of type 'NoneType' is not iterable"}
I am sorry, but I cannot process null values. Could you please provide the data in the correct format?


In [19]:
run_assistant("Check this data for duplicate IDs: [{\"id\": 1}, {\"id\": 2}]")

[TOOL] validate_data({'check': 'duplicate_ids', 'data': [{'id': 1}, {'id': 2}]})
[OK] {'ok': True, 'tool': 'validate_data', 'output': False}


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 17.64638029s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'q

### Runtime Error Evaluation

I actually encountered an earlier tool-use issue with `transform_data`, where the model did not make a tool call because the tool description was too general. I then made four additional attempts to produce another function-calling failure before reaching the rate limit.

The first two attempts did not produce a tool call because the model recognized that it needed the customer data. The third attempt successfully called `validate_data` with data that did not contain IDs and returned no duplicate IDs. The fourth attempt finally produced a runtime error when the model sent a null value inside the data array. The tool returned an `execution_error` because the function attempted to check for an ID in the null value.

As shown by these attempts, the constrained schema and tool descriptions worked well enough that the model avoided several potentially invalid calls. This made producing a real failure more difficult than expected, but the fourth attempt successfully exposed a runtime error. I reached the API rate limit before I could complete a retry with corrected data, so I was unable to demonstrate the full recovery step. The model did recognize the error and requested that the data be provided in the correct format, but I could not verify a successful retry before the rate limit.


## Part 5: Submit
Open a pull request with your schema design write-up, a link to your notebook, and a link to an issue documenting the failure and recovery. Describe your code-runner's allowlist, time limit, and blocked operations. Rubric: schemas (20), loop including a two-step sequence (25), guarded code-runner (20), failure with recovery (20), PR hygiene (15).